## Data Cleaning 
### Launch excel, merge sheets, clean nulls

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
# Load the Excel file to inspect sheets
file_path = 'Pharmacy_data.xlsx'
xls = pd.ExcelFile(file_path)
print(f"Sheet names: {xls.sheet_names}")

Sheet names: ['FactSales', 'DimDate', 'DimPharmacy', 'DimProduct']


In [4]:
# Read all sheets into a dictionary of dataframes
dfs = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in xls.sheet_names}

# Display basic info and head for each sheet to understand structure
for sheet, df in dfs.items():
    print(f"\n--- Sheet: {sheet} ---")
    print(df.info())
    print(df.head())


--- Sheet: FactSales ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62139 entries, 0 to 62138
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SalesID     62139 non-null  object 
 1   DateKey     62139 non-null  int64  
 2   PharmacyID  62139 non-null  object 
 3   ProductID   62139 non-null  object 
 4   UnitsSold   62139 non-null  int64  
 5   RevenueEUR  62139 non-null  float64
 6   CostEUR     62139 non-null  float64
 7   MarginEUR   62139 non-null  float64
 8   PromoFlag   62139 non-null  object 
dtypes: float64(3), int64(2), object(4)
memory usage: 4.3+ MB
None
    SalesID   DateKey PharmacyID ProductID  UnitsSold  RevenueEUR  CostEUR  \
0  S0000001  20240101     PH0002    PR0099          5      128.08    87.55   
1  S0000002  20240101     PH0004    PR0156          4       51.89    34.32   
2  S0000003  20240101     PH0007    PR0004         20      317.73   199.53   
3  S0000004  20240101     PH0009    

In [5]:
# Merge the dataframes to create a master dataset
merged_df = dfs['FactSales'].merge(dfs['DimDate'], on='DateKey', how='left')
merged_df = merged_df.merge(dfs['DimPharmacy'], on='PharmacyID', how='left')
merged_df = merged_df.merge(dfs['DimProduct'], on='ProductID', how='left')

In [6]:
# Check for Nulls
null_counts = merged_df.isnull().sum()
print(null_counts)

SalesID                 0
DateKey                 0
PharmacyID              0
ProductID               0
UnitsSold               0
RevenueEUR              0
CostEUR                 0
MarginEUR               0
PromoFlag               0
Date                    0
Year                    0
Quarter                 0
MonthNumber             0
MonthName               0
YearMonth               0
PharmacyName            0
Country                 0
Region                  0
City                    0
PharmacyType            0
OpenDate                0
StoreSizeBand           0
Latitude                0
Longitude               0
ProductName             0
Category                0
Brand                   0
IsGeneric               0
PackSize                0
ListPriceEUR            0
StandardCostEUR         0
LaunchDate              0
IsDiscontinued          0
DiscontinuedDate    57094
dtype: int64


In [7]:
for col in merged_df:

    print(col)
    

SalesID
DateKey
PharmacyID
ProductID
UnitsSold
RevenueEUR
CostEUR
MarginEUR
PromoFlag
Date
Year
Quarter
MonthNumber
MonthName
YearMonth
PharmacyName
Country
Region
City
PharmacyType
OpenDate
StoreSizeBand
Latitude
Longitude
ProductName
Category
Brand
IsGeneric
PackSize
ListPriceEUR
StandardCostEUR
LaunchDate
IsDiscontinued
DiscontinuedDate


In [8]:
cols_to_drop = ['SalesID', 'DateKey', 'PharmacyID', 'ProductID', 
    'Latitude', 'Longitude', 'LaunchDate', 'PharmacyType', 'MonthNumber', 
    'YearMonth', 'StoreSizeBand', 'PackSize']

merged_df = merged_df.drop(columns=cols_to_drop)

In [9]:
# Standardize PromoFlag to boolean for easier analysis
merged_df['PromoFlag'] = merged_df['PromoFlag'].map({'Yes': True, 'No': False})

In [10]:
# Handle missing values in DiscontinuedDate for a cleaner output
merged_df['DiscontinuedDate'] = merged_df['DiscontinuedDate'].fillna('N/A')

In [13]:
# Change date to date-time
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [14]:
cleaned_df = merged_df

cleaned_df.to_csv('Cleaned_Pharmacy_Sales.csv', index=False)

In [15]:
# Summary

summary = {
    "Total Records": len(cleaned_df),
    "Total Revenue": cleaned_df['RevenueEUR'].sum(),
    "Unique Products": cleaned_df['ProductName'].nunique(),
}
print(summary)
print(cleaned_df.head())

{'Total Records': 62139, 'Total Revenue': np.float64(8633977.31), 'Unique Products': 203}
   UnitsSold  RevenueEUR  CostEUR  MarginEUR  PromoFlag       Date  Year  \
0          5      128.08    87.55      40.53      False 2024-01-01  2024   
1          4       51.89    34.32      17.57      False 2024-01-01  2024   
2         20      317.73   199.53     118.20      False 2024-01-01  2024   
3          6       90.34    67.49      22.85      False 2024-01-01  2024   
4          2      160.21   124.99      35.22       True 2024-01-01  2024   

   Quarter MonthName                PharmacyName  ...       City    OpenDate  \
0        1   January  Innsbruck HealthPoint #002  ...  Innsbruck  2011-12-21   
1        1   January   Katowice HealthPoint #004  ...   Katowice  2018-07-18   
2        1   January    Cologne HealthPoint #007  ...    Cologne  2017-07-25   
3        1   January     Munich HealthPoint #009  ...     Munich  2010-08-03   
4        1   January      Liège HealthPoint #010  ...

## Feature Engineering 

In [16]:
# Staring from the cleaned dataset 

df = pd.read_csv('Cleaned_Pharmacy_Sales.csv')
df['Date'] = pd.to_datetime(df['Date'])


In [17]:
# 1. Margin Percentage: To understand profitability per transaction.

df['MarginPercentage'] = (df['MarginEUR'] / df['RevenueEUR']) * 100

print(df[['MarginPercentage']].head())


   MarginPercentage
0         31.644285
1         33.860089
2         37.201397
3         25.293336
4         21.983646


In [18]:
# 2. Seasonality: Extracting Month from Date for trend analysis.
df['Month'] = df['Date'].dt.month_name()

print(df[['Date', 'Month']].head())


        Date    Month
0 2024-01-01  January
1 2024-01-01  January
2 2024-01-01  January
3 2024-01-01  January
4 2024-01-01  January


In [19]:
# 3. Unit Profitability: Margin earned per individual unit sold.
df['MarginPerUnit'] = df['MarginEUR'] / df['UnitsSold']

print(df[['MarginPerUnit']].head())

   MarginPerUnit
0       8.106000
1       4.392500
2       5.910000
3       3.808333
4      17.610000


In [20]:
# analysis quaestions 

df.head(1)

# Reviewing possible analysis questions based on the available data after cleaning 

,UnitsSold,RevenueEUR,CostEUR,MarginEUR,PromoFlag,Date,Year,Quarter,MonthName,PharmacyName,...,Category,Brand,IsGeneric,ListPriceEUR,StandardCostEUR,IsDiscontinued,DiscontinuedDate,MarginPercentage,Month,MarginPerUnit
0,5,128.08,87.55,40.53,False,2024-01-01,2024,1,January,Innsbruck HealthPoint #002,...,Personal Care,HairEssence,No,24.63,17.25,No,NaN,31.644285,January,8.106


In [21]:
# 1. Top Brand per Country
brand_country = df.groupby(['Country', 'Brand'])['RevenueEUR'].sum().reset_index()
top_brands = brand_country.loc[brand_country.groupby(['Country'])['RevenueEUR'].idxmax()]
fig1 = px.bar(top_brands, x='Country', y='RevenueEUR', color='Brand', title="Q1 - Fig 1:Leading Brand by Country")
fig1

In [22]:
# 2. Top Selling Promo Items
promo_items = df[df['PromoFlag'] == True].groupby('ProductName')['UnitsSold'].sum().reset_index()
top_promo = promo_items.sort_values('UnitsSold', ascending=False).head(10)
fig2 = px.bar(top_promo, x='UnitsSold', y='ProductName', orientation='h', title="Q2 - Fig 2:Top Promo Sellers")
fig2

In [23]:
# 3. Generic vs Brand Margin Impact
generic_impact = df.groupby('IsGeneric')['MarginPercentage'].mean().reset_index()
fig3 = px.bar(generic_impact, x='IsGeneric', y='MarginPercentage', title="Q3- Fig 3:Profit Margin %: Generic vs Brand")
fig3

In [24]:
# 4. Revenue for Promo Flag Yes 
promo_split = df.groupby('PromoFlag')['RevenueEUR'].sum().reset_index()
fig4 = px.pie(promo_split, values='RevenueEUR', names='PromoFlag', title=" Q4 - Fig 4:Total Revenue: Promo vs Standard")
fig4

In [25]:
# Q5: Time Series Analysis of Sales per Month

monthly_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

monthly_sales = df.groupby('MonthName')['RevenueEUR'].sum().reindex(monthly_order).reset_index()

fig5 = px.line(
    monthly_sales, 
    x='MonthName', y='RevenueEUR', title="Q5 - Fig 5: Time Series Analysis of Sales per Month", 
    markers=True, line_shape="spline", template="plotly_white")
fig5

In [26]:
# Q6: Which product categories generate the highest total revenue? 

cat_rev = df.groupby('Category')['RevenueEUR'].sum().sort_values(ascending=False).reset_index()
fig6 = px.bar(
    cat_rev, x='Category', y='RevenueEUR', title="Q6 - Fig 6: Total Revenue by Product Category",
    labels={'RevenueEUR': 'Total Revenue (EUR)'}, color='RevenueEUR', color_continuous_scale='Viridis')

fig6

In [27]:
# Q7: How is the total Margin distributed across top Regions and Brands?

top_regions = df.groupby('Region')['MarginEUR'].sum().nlargest(5).index
top_brands = df.groupby('Brand')['MarginEUR'].sum().nlargest(5).index
df_filtered = df[df['Region'].isin(top_regions) & df['Brand'].isin(top_brands)]


fig7 = px.box(df_filtered, x='Region', y='MarginPercentage', color='Brand',
    title="Fig 7: Margin % Distribution by Region and Brand", labels={'MarginPercentage': 'Margin %'}, template='plotly_white')

fig7